# Lab 2 report



## LAB_INSTRUCTION.md

In `notebook_01_psycopg.ipynb`, I set up PostgreSQL, created the `services` table, and practiced querying it with Psycopg. I also added a query that exports Amsterdam Central trains with late arrivals and departures to JSONL; it returned no rows because the table had no matching data, then I copied the rows with psql command to the services table to postgres container. As the result 29396 rows were saved to data/late_train.jsonl

The `notebook_01_psycopg.ipynb` exercise was considerably more challenging than I expected. I spent around 2-3 hours troubleshooting table-creation and query errors, and checking that the data was loaded correctly before running queries.


# HOMEWORK.md

## Step 1: Download the datasets


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

download_items = {
    "stations-2023-09.csv": "https://opendata.rijdendetreinen.nl/public/stations/stations-2023-09.csv",
    "tariff-distances-2022-01.csv": "https://blobs.duckdb.org/data/tariff-distances-2022-01.csv",
}
download_items.update({
    f"disruptions-{year}.csv": f"https://opendata.rijdendetreinen.nl/public/disruptions/disruptions-{year}.csv"
    for year in range(2011, 2024)
})
download_items.update({
    f"services-{year}.csv.gz": f"https://opendata.rijdendetreinen.nl/public/services/services-{year}.csv.gz"
    for year in range(2019, 2026)
})

for filename, url in download_items.items():
    path = DATA_DIR / filename
    if path.is_file() and path.stat().st_size > 0:
        print(f"Already downloaded: {filename}")
        continue
    print(f"Downloading {filename}...")
    urlretrieve(url, path)
    print(f"Saved to {path}")

## Step 2: Load stations into DuckDB

In [ ]:
import duckdb

db = duckdb.connect("data/homework.duckdb")
db.execute("""
CREATE OR REPLACE TABLE stations AS
SELECT *
FROM read_csv_auto('data/stations-2023-09.csv', header = true)
""")

display(db.sql("DESCRIBE stations").df())
display(db.sql("SELECT COUNT(*) AS station_count FROM stations").df())

## Step 3: Create the distance tables


In [ ]:
db.execute("""
CREATE OR REPLACE TABLE distances AS
SELECT *
FROM read_csv('data/tariff-distances-2022-01.csv', header = true, nullstr = 'XXX')
""")

db.execute("""
CREATE OR REPLACE TABLE distances_long AS
UNPIVOT distances
ON COLUMNS (* EXCLUDE Station)
INTO NAME other_station VALUE distance
""")

display(db.sql("DESCRIBE distances").df().head())
display(db.sql("SELECT * FROM distances_long LIMIT 5").df())
display(db.sql("SELECT COUNT(*) AS station_rows FROM distances").df())
display(db.sql("SELECT COUNT(*) AS pair_rows FROM distances_long").df())

## Step 4: Load disruptions into Postgres


In [ ]:
import json
import subprocess

import psycopg
from psycopg import sql

compose_config = json.loads(subprocess.run(
    ["docker", "compose", "config", "--format", "json"],
    check=True,
    capture_output=True,
    text=True,
).stdout)
postgres_service = compose_config["services"]["postgres"]
postgres_env = postgres_service["environment"]
postgres_port = next(
    int(port["published"])
    for port in postgres_service["ports"]
    if int(port["target"]) == 5432
)
admin_conninfo = psycopg.conninfo.make_conninfo(
    host="localhost",
    port=postgres_port,
    user=postgres_env["POSTGRES_USER"],
    password=postgres_env["POSTGRES_PASSWORD"],
    dbname=postgres_env["POSTGRES_DB"],
)
homework_db_name = "lab02_homework"

with psycopg.connect(admin_conninfo, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (homework_db_name,))
        if cur.fetchone() is None:
            cur.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(homework_db_name)))

homework_conninfo = psycopg.conninfo.make_conninfo(admin_conninfo, dbname=homework_db_name)
disruption_files = [DATA_DIR / f"disruptions-{year}.csv" for year in range(2011, 2024)]
missing_files = [path for path in disruption_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing disruption files: {missing_files}")

with psycopg.connect(homework_conninfo) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS disruptions (
                rdt_id BIGINT,
                ns_lines TEXT,
                rdt_lines TEXT,
                rdt_lines_id TEXT,
                rdt_station_names TEXT,
                rdt_station_codes TEXT,
                cause_nl TEXT,
                cause_en TEXT,
                statistical_cause_nl TEXT,
                statistical_cause_en TEXT,
                cause_group TEXT,
                start_time TIMESTAMP,
                end_time TIMESTAMP,
                duration_minutes INTEGER
            )
        """)
        cur.execute("TRUNCATE TABLE disruptions")
        copy_sql = """
            COPY disruptions (
                rdt_id, ns_lines, rdt_lines, rdt_lines_id, rdt_station_names,
                rdt_station_codes, cause_nl, cause_en, statistical_cause_nl,
                statistical_cause_en, cause_group, start_time, end_time,
                duration_minutes
            ) FROM STDIN WITH (FORMAT CSV, HEADER TRUE)
        """
        with cur.copy(copy_sql) as copy:
            for file_index, path in enumerate(disruption_files):
                with path.open("rb") as source:
                    if file_index:
                        source.readline()  # Each annual CSV has the same header.
                    while chunk := source.read(1024 * 1024):
                        copy.write(chunk)

with psycopg.connect(homework_conninfo) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM disruptions")
        print(f"Loaded {cur.fetchone()[0]:,} disruptions into {homework_db_name}.disruptions")

## Step 5: Combine train services into Parquet

In [ ]:
service_files = sorted(DATA_DIR.glob("services-*.csv.gz"))
if not service_files:
    raise FileNotFoundError("No annual services-*.csv.gz files found in data/")

services_parquet = DATA_DIR / "services.parquet"
if not services_parquet.is_file() or services_parquet.stat().st_size == 0:
    service_file_list = ", ".join(
        "'" + path.as_posix().replace("'", "''") + "'" for path in service_files
    )
    db.execute(f"""
        COPY (
            SELECT *
            FROM read_csv(
                [{service_file_list}],
                header = true,
                auto_detect = true,
                union_by_name = true
            )
        ) TO '{services_parquet.as_posix()}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
else:
    print(f"Reusing existing {services_parquet}")

db.execute("""
    CREATE OR REPLACE TABLE services AS
    SELECT * FROM read_parquet('data/services.parquet')
""")
display(db.sql("SELECT COUNT(*) AS service_stop_rows FROM services").df())
print(f"Parquet size: {services_parquet.stat().st_size / (1024 ** 2):,.1f} MiB")